In [32]:
# imports

import pandas as pd
import numpy as np
import itertools
import uuid
import random

In [2]:
# loading data

df = pd.read_csv("../data/raw/Sensorimotor_norms_24Jan2026.csv")

In [ ]:
df.head()

In [ ]:
# separating out only the words that have haptic as their main modality (Dominant.perceptual column)

df_haptic = df[df["Dominant.perceptual"] == "Haptic"]


pd.set_option("display.max_rows", 975)
display(df_haptic)

In [ ]:
# filtering all the haptic words that have a score over 2 at least. for the seed set still

df_haptic_candidates = (
    df[df["Haptic.mean"] >= 2.0]
    .sort_values("Haptic.mean", ascending=False)
)

df_haptic_candidates.head()
len(df_haptic_candidates)

In [18]:
tierA = df_haptic_candidates[df_haptic_candidates["Haptic.mean"] >= 3.5]
tierB = df_haptic_candidates[(df_haptic_candidates["Haptic.mean"] >= 3.0) & (df_haptic_candidates["Haptic.mean"] < 3.5)]
tierC = df_haptic_candidates[(df_haptic_candidates["Haptic.mean"] >= 2.0) & (df_haptic_candidates["Haptic.mean"] < 3.0)]

In [19]:
tierA.to_csv("tierA_haptic_ge_3_5.csv", index=False)
tierB.to_csv("tierB_haptic_3_0_3_5.csv", index=False)

After inspection and careful consideration, this is the finalized seed set for the given modalities. These will anchor the generation process as well.

1. nociception: pain; painful; painfulness; burn; burning; scalding; blistering; sting; stinging; pricking; prick; pricked; pinprick; itch; itching; itchiness; bruise; jab; jabbed; puncture; pierced; injure; sharp; sharper; spiky; spikiness; thorny; thorniness; jagged; jaggedness; electroshock; electric shock; electrical shock; jolt; ache; aching; sore; soreness; hurt; hurting
2. temperature: hot; hotness; heat; warm; warmth; warmness; warmer; lukewarm; tepid; cool; coolness; cooler; cold; coldness; colder; coldish; freezing; frozen; freeze; frost; frostiness; icy; iciness; ice cold; overheated; superheated; heating; cooling; warming; unheated; room temperature; chilly; chilliness; chilled; chilling; frigid; frosty
3. vibration: vibration; vibrate; vibrating; pulse; pulsate; throbbing; trembling; oscillating; oscillation
4. pressure: pressure; press; pressing; press down; squeeze; squeezing; compress; compressible; tight; tightness; force; compression; tension

In [23]:
## Creating the seed dataframe

# defining the seed dictionary

seeds = {
    "nociception": [
        "PAIN","PAINFUL","PAINFULNESS","BURN","BURNING","SCALDING","BLISTERING",
        "STING","STINGING","PRICKING","PRICK","PRICKED","PINPRICK","ITCH","ITCHING",
        "ITCHINESS","BRUISE","JAB","JABBED","PUNCTURE","PIERCED","INJURE","SHARP",
        "SHARPER","SPIKY","SPIKINESS","THORNY","THORNINESS","JAGGED","JAGGEDNESS",
        "ELECTROSHOCK","ELECTRIC SHOCK","ELECTRICAL SHOCK","JOLT","ACHE","ACHING",
        "SORE","SORENESS","HURT","HURTING"
    ],
    "temperature": [
        "HOT","HOTNESS","HEAT","WARM","WARMTH","WARMNESS","WARMER","LUKEWARM","TEPID",
        "COOL","COOLNESS","COOLER","COLD","COLDNESS","COLDER","COLDISH","FREEZING",
        "FROZEN","FREEZE","FROST","FROSTINESS","ICY","ICINESS","ICE COLD","OVERHEATED",
        "SUPERHEATED","HEATING","COOLING","WARMING","UNHEATED","ROOM TEMPERATURE",
        "CHILLY","CHILLINESS","CHILLED","CHILLING","FRIGID","FROSTY"
    ],
    "vibration": [
        "VIBRATION","VIBRATE","VIBRATING","PULSE","PULSATE",
        "THROBBING","TREMBLING","OSCILLATING"
    ],
    "pressure": [
        "PRESSURE","PRESS","PRESSING","PRESS DOWN","SQUEEZE","SQUEEZING",
        "COMPRESS","COMPRESSIBLE","TIGHT","TIGHTNESS","FORCE",
        "COMPRESSION","TENSION"
    ]
}



# flattening seeds into a lookup table 

seed_lookup = {}

for modality, words in seeds.items():
    for w in words:
        seed_lookup[w] = modality



# filtering the original dataframe 

df_seeds = df[df["Word"].isin(seed_lookup.keys())].copy()


# adding "Modality" and "IsSeed" columns

df_seeds["Modality"] = df_seeds["Word"].map(seed_lookup)
df_seeds["IsSeed"] = 1



In [ ]:
df_seeds[["Word", "Modality"]].head(20)

In [25]:
missing = set(seed_lookup.keys()) - set(df_seeds["Word"])
print(missing)

set()


In [ ]:
pd.set_option("display.max_rows", 975)
display(df_seeds)

In [ ]:
# doing a dist check of the seed word distributions

seed_dist = (
    df_seeds
    .groupby("Modality")
    .size()
    .reset_index(name="n_seeds")
)

seed_dist["proportion"] = seed_dist["n_seeds"] / seed_dist["n_seeds"].sum()
seed_dist


In [ ]:
seed_dist.plot(
    x="Modality",
    y="n_seeds",
    kind="bar",
    legend=False,
    title="Seed word distribution by modality")

In [ ]:
### creating the generation plan table (UPDATED: x=36 weighting + multiple anchor variants)

import itertools, random, uuid
import pandas as pd

## defining modalities
modalities = ["nociception", "temperature", "vibration", "pressure"]

# single, pair, triple combinations
single_labels = [(m,) for m in modalities]
pair_labels   = list(itertools.combinations(modalities, 2))
triple_labels = list(itertools.combinations(modalities, 3))

label_combinations = single_labels + pair_labels + triple_labels

## defining generation axes
literal_levels      = [0, 1]        # 0=metaphorical, 1=literal
specificity_levels  = [0, 1]        # 0=vague, 1=specific
consistency_levels  = [0, 1]        # 0=inconsistent, 1=consistent
anchor_regimes      = ["strict", "paraphrase", "drift"]

## defining anchors per modality (locked seed list)
anchors_by_modality = {
    "nociception": [
        "PAIN","PAINFUL","PAINFULNESS","BURN","BURNING","SCALDING","BLISTERING",
        "STING","STINGING","PRICKING","PRICK","PRICKED","PINPRICK","ITCH","ITCHING",
        "ITCHINESS","BRUISE","JAB","JABBED","PUNCTURE","PIERCED","INJURE","SHARP",
        "SHARPER","SPIKY","SPIKINESS","THORNY","THORNINESS","JAGGED","JAGGEDNESS",
        "ELECTROSHOCK","ELECTRIC SHOCK","ELECTRICAL SHOCK","JOLT","ACHE","ACHING",
        "SORE","SORENESS","HURT","HURTING"
    ],
    "temperature": [
        "HOT","HOTNESS","HEAT","WARM","WARMTH","WARMNESS","WARMER","LUKEWARM","TEPID",
        "COOL","COOLNESS","COOLER","COLD","COLDNESS","COLDER","COLDISH","FREEZING",
        "FROZEN","FREEZE","FROST","FROSTINESS","ICY","ICINESS","ICE COLD","OVERHEATED",
        "SUPERHEATED","HEATING","COOLING","WARMING","UNHEATED","ROOM TEMPERATURE",
        "CHILLY","CHILLINESS","CHILLED","CHILLING","FRIGID","FROSTY"
    ],
    "vibration": [
        "VIBRATION","VIBRATE","VIBRATING","PULSE","PULSATE",
        "THROBBING","TREMBLING","OSCILLATING"
    ],
    "pressure": [
        "PRESSURE","PRESS","PRESSING","PRESS DOWN","SQUEEZE","SQUEEZING",
        "COMPRESS","COMPRESSIBLE","TIGHT","TIGHTNESS","FORCE",
        "COMPRESSION","TENSION"
    ]
}

## Helper functions
def format_labels(labels):
    return "+".join(labels)

def select_anchors(labels):
    anchors = []
    for label in labels:
        anchors.append(random.choice(anchors_by_modality[label]))
    return "; ".join(anchors)

# weighting rule (x, x/2, x/4)
X = 36
def total_samples_for_labels(labels_tuple):
    k = len(labels_tuple)
    if k == 1:
        return X
    if k == 2:
        return X // 2
    if k == 3:
        return X // 4
    raise ValueError(f"Unexpected label length {k}. Expected 1,2,3 only.")

# anchor-variant replication to reduce repetition
ANCHOR_VARIANTS = 3

# reproducibility
random.seed(42)

## building generation plan table
rows = []

for labels in label_combinations:
    n_total = total_samples_for_labels(labels)

    for literal, specificity, consistent, anchor_regime in itertools.product(
        literal_levels, specificity_levels, consistency_levels, anchor_regimes
    ):
        for rep in range(ANCHOR_VARIANTS):
            if anchor_regime == "drift":
                anchors = ""
            else:
                anchors = select_anchors(labels)

            n_samples = (n_total // ANCHOR_VARIANTS) + (1 if rep < (n_total % ANCHOR_VARIANTS) else 0)

            rows.append({
                "job_id": str(uuid.uuid4()),
                "generator": None, 
                "labels": format_labels(labels),
                "literal": literal,
                "specificity": specificity,
                "consistent": consistent,
                "anchor_regime": anchor_regime,
                "anchors": anchors,
                "n_samples": n_samples
            })

generation_plan = pd.DataFrame(rows)

## assigning jobs evenly to OpenAI and Claude
generators = ["openai", "claude"]
generation_plan = generation_plan.sample(frac=1, random_state=42).reset_index(drop=True)
generation_plan["generator"] = [generators[i % len(generators)] for i in range(len(generation_plan))]

## saving
generation_plan.to_csv("generation_plan.csv", index=False)

generation_plan


In [ ]:
## sanity checking the generation plan table - all good:)

generation_plan.groupby("generator").size()
generation_plan.groupby("anchor_regime").size()
generation_plan.groupby("labels").size()
